# Train, Evaluate, and Compare -- Google Colab

## 1. Overview and research question

**Research question.** Does a shared backbone with task-specific adapters
and learned loss balancing reduce negative transfer relative to naive
sharing or fully independent backbones? And does a residual (skip-
connection) design actually help, once measured against a depth/width-
matched non-residual baseline rather than an unrelated compact CNN?

**What this notebook runs, in one pass.** The full controlled ablation
suite: Experiment 0 (SimpleCNN), 0b (PlainDeep18NoSkip -- the residual-
connection control), 0c (no-zero-init-residual control), A (separate
backbones), B (shared, no adapters), C (shared + adapters, fixed
weights), and D (shared + adapters + learned balancing). Then Experiment
E (D's parametric heads vs. a k-NN baseline over D's own embeddings).
Then, optionally: non-parametric raw-pixel/frozen-backbone baselines,
robustness testing under 11 corruption types, Grad-CAM attention maps,
and multi-seed aggregation -- all controlled by toggles in the
configuration cell below.

**What this notebook is.** An orchestration layer around this
repository's real code. Every step below calls the actual scripts under
`scripts/` (which in turn call `src/`) exactly as they exist in the
repository -- nothing here reimplements model, dataset, loss, training,
or inference logic.

**Ethical framing (kept intact throughout).** The "gender-label" task
predicts a categorical field defined by the source dataset's own
documentation -- it is **not** a determination of a person's gender
identity. This project is research/education only and must not be used
for employment, policing, surveillance, identity verification, medical
diagnosis, admissions, insurance, or any other high-impact decision.

**Resume-safety, honestly stated.** Checkpoints saved by this repository
do not include optimizer state, so re-running this notebook after an
interruption *skips* any experiment/seed that already has a complete
checkpoint + test metrics -- it does not resume a training run
mid-epoch. Every stage (train / calibrate / build k-NN index / evaluate)
is checked independently, so a crash partway through still leaves every
completed stage intact for next time. Set `USE_GOOGLE_DRIVE = True`
below to sync progress to Drive after every stage, so a Colab session
timeout never loses completed work.

## 2. User configuration

In [ ]:
# ============================================================
# USER CONFIGURATION -- edit this cell, then run the notebook top to bottom.
# ============================================================

# --- What to run -----------------------------------------------------
# The full ablation suite (0, 0b, 0c, A, B, C, D) and Experiment E always
# run. These toggles control everything else.
SMOKE_TEST = False  # True = 1 epoch/3 batches per experiment, fast pipeline check only, NOT scientific results
FORCE_RERUN = False  # True = retrain everything from scratch, ignoring existing checkpoints
ALLOW_TEST_FAILURES = False

RUN_EXPERIMENT_E = True           # parametric heads vs. k-NN over D's embeddings
RUN_NONPARAMETRIC_BASELINES = True  # raw-pixel/PCA + frozen-backbone k-NN/KDE baselines (CPU-only)
RUN_ROBUSTNESS = True             # corruption-robustness sweep for 0 / 0b / D
RUN_GRADCAM = True                # Grad-CAM attention maps for 0 / 0b / D
RUN_MULTI_SEED = False            # re-run 0 / 0b / D at every seed in SEEDS below (multiplies training time)

# Pre-registered seeds for multi-seed runs (docs/final_evaluation_protocol.md).
# SEEDS[0] is always used for the single-seed run regardless of RUN_MULTI_SEED.
SEEDS = [42, 123, 1729]
MAX_EPOCHS = 40  # ceiling per stage/warm-up; early stopping (below) usually stops sooner
EARLY_STOPPING_PATIENCE = 12

# Optional hard cap on batches-per-epoch, independent of MAX_EPOCHS -- None
# (default) means no limit. SMOKE_TEST overrides this to a small number.
MAX_BATCHES_PER_EPOCH = None

# --- Training-quality settings (configs/training.yaml / configs/model.yaml
# defaults, exposed here for convenience) -- applied identically to every
# experiment, so none of this changes which architecture the results say
# is better; it only improves training quality across the board.
# ------------------------------------------------------------------------
# Differential (discriminative) learning rates: the backbone trains at
# `stage_lr * BACKBONE_LR_MULTIPLIER` while adapters/heads train at the
# full `stage_lr`.
DIFFERENTIAL_LR_ENABLED = True
BACKBONE_LR_MULTIPLIER = 0.1  # 1.0 disables the differential effect (same LR everywhere)
ADAPTER_BOTTLENECK_DIM = 256
# Epochs of equal (1.0/1.0) fixed loss weighting before the
# learned-uncertainty mechanism (log-variance weights) takes over.
LOSS_BALANCING_WARMUP_EPOCHS = 3

# --- Where backbone selection and the gender-label confidence threshold
# (tau) actually live: NOT in this cell -- see configs/model.yaml and
# configs/experiments.yaml. This notebook only chooses *how long* each
# experiment trains, never redefines its architecture inline.

REPO_URL = "https://github.com/adischwartz15/AgeGender.git"
REPO_BRANCH = "remove/volo-pretrained-transfer-learning"  # TODO: change back to "main" once this branch is merged

# Set to an existing run's RUN_ID (printed by a previous execution of this
# notebook, e.g. "2026-07-07_1430_seed42") to continue that run instead
# of starting a new one. Leave None for a fresh run -- a fresh run NEVER
# overwrites an existing run directory (see the run-ID cell below).
RESUME_RUN_ID = None

USE_GOOGLE_DRIVE = True

# Set this if you'd rather download the dataset via the Kaggle API from
# within Colab instead of using Drive (see notebooks/train_evaluate_kaggle.ipynb
# for a Kaggle-hosted alternative that has this pre-filled).
KAGGLE_DATASET_SLUG = None

# Set this only if the dataset is already available in Drive.
DRIVE_DATASET_DIR = None

if FORCE_RERUN:
    from IPython.display import Markdown, display
    display(Markdown(
        "> **Full clean rerun enabled**\n>\n"
        "> All reported results in this run are generated from newly trained "
        "models\n> and will not reuse prior checkpoints or metrics."
    ))

In [ ]:
# ============================================================
# Helper library -- shared plumbing used by every phase below.
# None of this reimplements model/dataset/training/evaluation logic; it only
# runs the repository's own scripts, and manages files/manifests around them.
# ============================================================

import datetime
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from IPython.display import Image, Markdown, display


def run_command(cmd, cwd=None, log_path=None, check=True, env=None):
    """Run a subprocess, streaming output live and optionally capturing it to a log file."""
    printable = cmd if isinstance(cmd, str) else " ".join(str(c) for c in cmd)
    print(f"$ {printable}")
    full_env = {**os.environ, **(env or {})}
    proc = subprocess.Popen(
        cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, shell=isinstance(cmd, str), env=full_env,
    )
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    proc.wait()
    if log_path:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text("".join(lines), encoding="utf-8")
    if check and proc.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {proc.returncode}): {printable}"
            + (f"\nSee log: {log_path}" if log_path else "")
        )
    return proc.returncode, "".join(lines)


def write_manifest(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(data, fh, indent=2, default=str)
    return path


def load_json(path):
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def safe_copy2(src, dst):
    """shutil.copy2, but skips (with a message) instead of raising when src and dst resolve to the same file.

    This happens on a resumed run whose RUN_DIR already points directly at
    the persistent Drive/output run directory (e.g. a Colab run resumed
    from a Drive-backed RUN_DIR): the log/checkpoint/metrics files under
    RUN_DIR are already being written to their final destination, so a
    later "sync to persistent storage" copy step would otherwise raise
    shutil.SameFileError trying to copy a file onto itself.
    """
    src, dst = Path(src), Path(dst)
    if src.resolve() == dst.resolve():
        print(f"Skipping copy: source and destination are the same file ({dst})")
        return dst
    return shutil.copy2(src, dst)


def copy_tree_merge(src, dst):
    """Copy every file under src into dst (creating dst), without touching unrelated existing dst content."""
    src, dst = Path(src), Path(dst)
    if not src.exists():
        return []
    dst.mkdir(parents=True, exist_ok=True)
    copied = []
    for path in src.rglob("*"):
        if path.is_file():
            target = dst / path.relative_to(src)
            target.parent.mkdir(parents=True, exist_ok=True)
            safe_copy2(path, target)
            copied.append(target)
    return copied


def move_tree_clear(src, dst):
    """Move every file under src into dst, then remove the now-empty src.

    Used for the couple of scripts (generate_gradcam.py, run_robustness.py)
    whose output directory is not configurable per run, so consecutive
    experiments would otherwise collide.
    """
    copied = copy_tree_merge(src, dst)
    if Path(src).exists():
        shutil.rmtree(src)
    return copied


def flatten_overrides(obj, prefix=""):
    """Flatten a nested dict into a flat list of ["--set", "a.b.c=value", ...] CLI args.

    Mirrors src/utils/config.py's own --set parsing (parse_cli_overrides /
    _coerce_scalar) exactly, so every value round-trips the same way it
    would from a hand-typed CLI override.
    """
    args = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            new_prefix = f"{prefix}.{key}" if prefix else key
            args.extend(flatten_overrides(value, new_prefix))
    else:
        value = obj
        if value is None:
            value = "null"
        elif isinstance(value, bool):
            value = "true" if value else "false"
        args += ["--set", f"{prefix}={value}"]
    return args


def display_image(path, caption=None):
    path = Path(path)
    if not path.exists():
        print(f"(plot not available: {path})")
        return
    if caption:
        display(Markdown(f"**{caption}**"))
    display(Image(filename=str(path)))


def artifact_ready(path):
    path = Path(path)
    return path.exists() and path.stat().st_size > 0


def validate_required_artifacts(paths):
    missing = [str(p) for p in paths if not artifact_ready(p)]
    if missing:
        raise RuntimeError("Required artifact(s) missing or empty:\n  " + "\n  ".join(missing))
    return True


print("Helper library loaded.")

## 3. Runtime and GPU verification

In [ ]:
# ============================================================
# Detect Google Colab
# ============================================================

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in Google Colab: {IN_COLAB}")
if not IN_COLAB:
    print(
        "This notebook is designed for Google Colab. It may still run "
        "elsewhere, but paths under /content are Colab-specific; consider "
        "notebooks/train_evaluate_kaggle.ipynb or a local checkout instead."
    )

In [ ]:
# ============================================================
# Runtime and GPU verification
# ============================================================

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU memory (GB): {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")
    else:
        print(
            "No GPU detected -- training will run on CPU and will be substantially "
            "slower. Enable a GPU accelerator in this platform's runtime settings "
            "before running the training cells below."
        )
except ImportError:
    print("PyTorch not yet installed -- run the dependency installation cell below first.")

## 4. Repository setup

In [ ]:
# ============================================================
# Repository setup
# ============================================================

REPO_DIR = Path("/content/AgeGender")

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    print(f"Repository already present at {REPO_DIR}; fetching latest {REPO_BRANCH}...")
    run_command(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_DIR)
    run_command(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR)
    run_command(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_DIR)
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_command(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")

## 5. Dependency installation

In [ ]:
# ============================================================
# Dependency installation
# ============================================================

run_command(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=REPO_DIR,
)

# Editable install is a convenience for `import src...` directly from notebook
# cells; every script under scripts/ already inserts the repo root onto its
# own sys.path and does not depend on this succeeding. pyproject.toml also
# currently declares requires-python >=3.11, so this is skipped gracefully
# (not treated as fatal) on older interpreters.
try:
    run_command([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=REPO_DIR)
    print("Editable install succeeded.")
except Exception as exc:
    print(f"Editable install skipped ({exc}); continuing with sys.path only (this is expected and fine).")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import torch  # noqa: E402  (re-import after installation, in case this is a fresh runtime)

print("Dependencies installed.")

## 6. Run ID, run directory, and environment manifest

Local workspace: `/content/agegender_runs/<RUN_ID>`. Persistent Drive destination (when `USE_GOOGLE_DRIVE=True`): `/content/drive/MyDrive/AgeGender/runs/<RUN_ID>`.

**Note:** `RUN_DIR` is created here, before the tests run below, since their log files live under it.

In [ ]:
# ============================================================
# Run ID and run directory -- a RESUME_RUN_ID does NOT require the local
# directory to already exist (a fresh Colab VM never has it); it is
# (re)created here and, if USE_GOOGLE_DRIVE=True, restored from Drive in
# the next cell. Restart-safety then comes from the per-stage checkpoint/
# metrics checks below finding what was restored.
# ============================================================

WORKSPACE_DIR = Path("/content/agegender_runs")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)


def _generate_run_id(seed):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
    return f"{timestamp}_seed{seed}"


PRIMARY_SEED = SEEDS[0]

if RESUME_RUN_ID:
    RUN_ID = RESUME_RUN_ID
    RUN_DIR = WORKSPACE_DIR / RUN_ID
    print(f"Resuming run: {RUN_DIR} (restored from Drive in the next cell, if USE_GOOGLE_DRIVE=True)")
else:
    candidate_id = _generate_run_id(PRIMARY_SEED)
    candidate_dir = WORKSPACE_DIR / candidate_id
    suffix = 1
    # Never overwrite an existing run directory silently.
    while candidate_dir.exists():
        suffix += 1
        candidate_dir = WORKSPACE_DIR / f"{candidate_id}_{suffix}"
    RUN_ID, RUN_DIR = candidate_dir.name, candidate_dir
    print(f"Starting new run: {RUN_DIR}")

RUN_SUBDIRS = [
    "config", "logs", "tests", "data_quality", "checkpoints", "metrics", "plots",
    "calibration", "gradcam", "robustness", "knn", "reports", "manifests",
    "archives", "experiments",
]
for _sub in RUN_SUBDIRS:
    (RUN_DIR / _sub).mkdir(parents=True, exist_ok=True)

print(f"RUN_ID  = {RUN_ID}")
print(f"RUN_DIR = {RUN_DIR}")

In [ ]:
# ============================================================
# Mount Google Drive (only if USE_GOOGLE_DRIVE=True) and, when resuming
# an existing RUN_ID, restore whatever was already synced there into this
# fresh runtime's local RUN_DIR -- a new Colab VM never has yesterday's
# /content, only Drive does, so without this step RESUME_RUN_ID would
# silently start every experiment over instead of skipping completed ones.
# ============================================================

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted at /content/drive")

    _drive_run_dir = Path("/content/drive/MyDrive/AgeGender/runs") / RUN_ID
    if _drive_run_dir.exists():
        for _sub in RUN_SUBDIRS:
            _restored = copy_tree_merge(_drive_run_dir / _sub, RUN_DIR / _sub)
            if _restored:
                print(f"Restored {len(_restored)} file(s) into {RUN_DIR / _sub} from Drive.")
    elif RESUME_RUN_ID:
        print(
            f"WARNING: RESUME_RUN_ID='{RESUME_RUN_ID}' was set but no Drive folder "
            f"found at {_drive_run_dir} -- nothing to restore, every experiment will "
            "run from scratch. Check the run ID (see a previous execution's printed "
            "RUN_DIR) if this is unexpected."
        )
else:
    print("Google Drive not mounted (USE_GOOGLE_DRIVE=False or not running in Colab).")


def sync_after_phase(phase_label):
    """Copy the run directory to Drive after a major phase, when enabled."""
    if IN_COLAB and USE_GOOGLE_DRIVE:
        dest = Path("/content/drive/MyDrive/AgeGender/runs") / RUN_ID
        copy_tree_merge(RUN_DIR, dest)
        print(f"[drive sync: {phase_label}] -> {dest}")

In [ ]:
# ============================================================
# Reproducibility and environment manifest
# (never includes credentials -- git status/log and package versions only)
# ============================================================

def _get_git_info(repo_dir):
    try:
        sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True).strip()
        branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=repo_dir, text=True).strip()
        status = subprocess.check_output(["git", "status", "--short"], cwd=repo_dir, text=True)
    except Exception as exc:
        sha, branch, status = f"unavailable ({exc})", "unavailable", ""
    return sha, branch, status


git_sha, git_branch, git_status = _get_git_info(REPO_DIR)

gpu_info = {}
if torch.cuda.is_available():
    gpu_info = {
        "name": torch.cuda.get_device_name(0),
        "total_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
        "cuda_version": torch.version.cuda,
    }

environment_manifest = {
    "run_id": RUN_ID,
    "git_commit_sha": git_sha,
    "git_branch": git_branch,
    "git_status_short": git_status,
    "python_version": sys.version,
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": gpu_info.get("cuda_version"),
    "gpu": gpu_info,
    "platform": platform.platform(),
    "smoke_test": SMOKE_TEST,
    "force_rerun": FORCE_RERUN,
    "generated_at_utc": datetime.datetime.utcnow().isoformat() + "Z",
}
write_manifest(RUN_DIR / "manifests" / "environment.json", environment_manifest)
print(json.dumps(environment_manifest, indent=2))

In [ ]:
sync_after_phase("repository_setup")

## 7. Dataset setup

Do not copy raw dataset images to Drive by default -- only metadata, split files, configs, logs, checkpoints, metrics, plots, reports, and archives are synchronized.

In [ ]:
# ============================================================
# Kaggle credentials (only needed if KAGGLE_DATASET_SLUG is set below)
# Never printed, logged, or written to a manifest.
# ============================================================

if KAGGLE_DATASET_SLUG and not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        print("Loaded Kaggle credentials from Colab Secrets.")
    except Exception:
        import getpass
        print("Enter Kaggle API credentials (hidden input; never printed or logged):")
        os.environ["KAGGLE_USERNAME"] = getpass.getpass("KAGGLE_USERNAME: ")
        os.environ["KAGGLE_KEY"] = getpass.getpass("KAGGLE_KEY: ")
elif KAGGLE_DATASET_SLUG:
    print("Kaggle credentials already present in the environment.")
else:
    print("KAGGLE_DATASET_SLUG not set; skipping (DRIVE_DATASET_DIR will be used instead).")

In [ ]:
# ============================================================
# Dataset setup
# ============================================================

if DRIVE_DATASET_DIR:
    dest = REPO_DIR / "data" / "raw"
    dest.mkdir(parents=True, exist_ok=True)
    print(f"Using dataset already available in Drive: {DRIVE_DATASET_DIR}")
    copy_tree_merge(DRIVE_DATASET_DIR, dest)
elif KAGGLE_DATASET_SLUG:
    os.environ["KAGGLE_DATASET_SLUG"] = KAGGLE_DATASET_SLUG
    if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        raise RuntimeError(
            "KAGGLE_USERNAME/KAGGLE_KEY are not set. Run the credentials cell "
            "above (Colab Secrets or hidden prompt) before this cell."
        )
    run_command(
        [sys.executable, "-u", "scripts/download_kaggle_data.py"],
        cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "download_data.log",
    )
else:
    raise RuntimeError(
        "No dataset source configured. Set KAGGLE_DATASET_SLUG (with Kaggle "
        "credentials available) or DRIVE_DATASET_DIR in the configuration "
        "cell before running this notebook."
    )

sync_after_phase("dataset_setup")

## 8. Data validation and deterministic split preparation

In [ ]:
# ============================================================
# Data validation and deterministic split preparation
# ============================================================

split_csv_path = RUN_DIR / "data_quality" / "full_metadata_with_splits.csv"
prepare_overrides = [
    "--set", f"paths.splits_dir={(RUN_DIR / 'data_quality').as_posix()}",
    "--set", f"validation.report_dir={(RUN_DIR / 'data_quality').as_posix()}",
]

if split_csv_path.exists() and not FORCE_RERUN:
    print(f"Reusing existing prepared split at {split_csv_path} (FORCE_RERUN=False).")
else:
    run_command(
        [sys.executable, "-u", "scripts/prepare_data.py"] + prepare_overrides,
        cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "prepare_data.log",
    )

quality_report = load_json(RUN_DIR / "data_quality" / "data_quality_report.json")
split_hash = sha256_file(split_csv_path)

print("Data quality report:")
print(json.dumps(quality_report, indent=2))
print(f"\nSplit file SHA-256: {split_hash}")

write_manifest(
    RUN_DIR / "manifests" / "data_manifest.json",
    {**quality_report, "split_file_sha256": split_hash, "split_file": str(split_csv_path)},
)
sync_after_phase("data_preparation")

## 9. Automated tests

In [ ]:
# ============================================================
# Automated tests (must pass, or ALLOW_TEST_FAILURES=True, before training)
# ============================================================

pytest_log = RUN_DIR / "tests" / "pytest.log"
junit_path = RUN_DIR / "tests" / "pytest_junit.xml"

returncode, _ = run_command(
    [sys.executable, "-m", "pytest", "-q", f"--junitxml={junit_path}"],
    cwd=REPO_DIR, log_path=pytest_log, check=False,
)
tests_passed = returncode == 0

print(f"\nPython tests: {'PASS' if tests_passed else 'FAIL'}")

if not tests_passed and not ALLOW_TEST_FAILURES:
    raise RuntimeError(
        "Automated tests failed and ALLOW_TEST_FAILURES=False. Fix the failures "
        f"(see {pytest_log}) before running expensive training, or set "
        "ALLOW_TEST_FAILURES=True in the configuration cell to proceed anyway "
        "(not recommended before a reported research run)."
    )

In [ ]:
sync_after_phase("tests")

## 10. Experiment selection (full architecture ablation suite)

In [ ]:
# ============================================================
# Experiment selection -- the full controlled ablation suite always runs:
# Experiments 0 (SimpleCNN), 0b (PlainDeep18NoSkip -- residual-connection
# control), 0c (no-zero-init-residual control), A (separate backbones),
# B (shared, no adapters), C (shared + adapters, fixed weights), and
# D (shared + adapters + learned balancing). Already-complete experiments
# (checkpoint + metrics on disk) are skipped automatically -- see the
# training helpers below.
# ============================================================

import yaml

CNN_EXPERIMENT = "exp_0_simple_cnn_shared_adapters_learned_balance"
PLAIN_DEEP18_EXPERIMENT = "exp_0b_plain_deep18_no_skip_shared_adapters_learned_balance"
NO_ZERO_INIT_EXPERIMENT = "exp_0c_custom_resnet18_no_zero_init_shared_adapters_learned_balance"
SEPARATE_EXPERIMENT = "exp_a_separate"
SHARED_NO_ADAPTERS_EXPERIMENT = "exp_b_shared_no_adapters"
SHARED_ADAPTERS_EXPERIMENT = "exp_c_shared_adapters"
RESNET_EXPERIMENT = "exp_d_shared_adapters_learned_balance"

experiments_cfg = yaml.safe_load((REPO_DIR / "configs" / "experiments.yaml").read_text(encoding="utf-8"))["experiments"]

ALL_CORE_EXPERIMENTS = [
    CNN_EXPERIMENT, PLAIN_DEEP18_EXPERIMENT, NO_ZERO_INIT_EXPERIMENT,
    SEPARATE_EXPERIMENT, SHARED_NO_ADAPTERS_EXPERIMENT, SHARED_ADAPTERS_EXPERIMENT,
    RESNET_EXPERIMENT,
]
missing = [name for name in ALL_CORE_EXPERIMENTS if name not in experiments_cfg]
if missing:
    raise RuntimeError(
        f"Experiment(s) not found in configs/experiments.yaml: {missing}. Stopping rather "
        "than silently running a different set of experiments than requested."
    )
selected_experiments = ALL_CORE_EXPERIMENTS
print(f"Will train/evaluate (already-complete ones are skipped, not retrained): {selected_experiments}")

if SMOKE_TEST:
    MAX_EPOCHS = min(MAX_EPOCHS, 1)
    EARLY_STOPPING_PATIENCE = min(EARLY_STOPPING_PATIENCE, 1)
    MAX_BATCHES_PER_EPOCH = MAX_BATCHES_PER_EPOCH or 3
    print(
        f"SMOKE_TEST=True: capping MAX_EPOCHS/EARLY_STOPPING_PATIENCE to "
        f"{MAX_EPOCHS}/{EARLY_STOPPING_PATIENCE} and batches-per-epoch to "
        f"{MAX_BATCHES_PER_EPOCH} for a fast integration check only. These "
        "results are NOT final scientific findings -- the comparison "
        "tables and detailed report below are skipped for this run."
    )

## 11. Training

Each experiment below is trained, calibrated, evaluated, and synchronized to
Drive in turn -- so a Colab session timeout after experiment *N* still
leaves experiments `1..N` fully persisted in Drive. Re-running this notebook
(same `RUN_ID`, or set `RESUME_RUN_ID`) will skip already-complete
experiments unless `FORCE_RERUN=True`.

In [ ]:
# ============================================================
# Training helpers (orchestration only -- calls scripts/train.py,
# scripts/calibrate.py, scripts/build_knn_index.py, scripts/evaluate.py)
#
# Each stage (train / calibrate / build k-NN index / evaluate) is checked
# and skipped independently, based only on whether *that stage's own*
# artifact already exists. This is what makes the pipeline stage-level
# restart-safe: if evaluation crashes after a successful training run, only
# evaluation reruns next time -- training is never redone just because a
# later stage failed.
# ============================================================

def experiment_paths(run_dir, experiment_name, seed):
    base = run_dir / "experiments" / experiment_name / f"seed_{seed}"
    return {
        "base": base,
        "checkpoint_dir": base / "checkpoints",
        "output_dir": base,
        "calibration_dir": base / "calibration",
        "knn_dir": base / "knn",
        "config_dir": base / "config",
        "log_dir": base / "logs",
    }


def _stage_status(artifact_path, force_rerun):
    exists = artifact_path.exists()
    if exists and not force_rerun:
        return "skipped", f"skipped, found at {artifact_path}"
    if exists and force_rerun:
        return "rerun", f"rerunning (FORCE_RERUN=True), would have reused {artifact_path}"
    return "run", "running, artifact missing"


def print_stage_plan(experiment_name, seed, paths, include_knn):
    """Print the skip/run decision for every stage before executing any of
    them, so a restart's behavior is transparent up front."""
    checkpoint = paths["checkpoint_dir"] / f"{experiment_name}_best_balanced_score.pt"
    calibration_artifact = paths["calibration_dir"] / "conformal_calibration.json"
    knn_index = paths["knn_dir"] / "knn_baseline.pkl"
    metrics_path = paths["output_dir"] / "metrics" / f"{experiment_name}_test_metrics.json"

    print(f"[{experiment_name} seed={seed}] Stage plan:")
    print(f"  - training:   {_stage_status(checkpoint, FORCE_RERUN)[1]}")
    print(f"  - calibration: {_stage_status(calibration_artifact, FORCE_RERUN)[1]}")
    if include_knn:
        print(f"  - k-NN index: {_stage_status(knn_index, FORCE_RERUN)[1]}")
    else:
        print("  - k-NN index: not requested for this experiment")
    print(f"  - evaluation: {_stage_status(metrics_path, FORCE_RERUN)[1]}")


def train_one_experiment(experiment_name, seed):
    spec = experiments_cfg[experiment_name]
    paths = experiment_paths(RUN_DIR, experiment_name, seed)
    for value in paths.values():
        value.mkdir(parents=True, exist_ok=True)

    checkpoint = paths["checkpoint_dir"] / f"{experiment_name}_best_balanced_score.pt"
    if checkpoint.exists() and not FORCE_RERUN:
        print(f"[{experiment_name} seed={seed}] training: skipped, checkpoint found ({checkpoint})")
        return paths

    overrides_args = flatten_overrides(spec.get("overrides", {}))
    notebook_overrides = {
        "seed": seed,
        "training": {
            "seed": seed,
            "warm_up_from_scratch": {"epochs": MAX_EPOCHS},
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "max_train_batches_per_epoch": MAX_BATCHES_PER_EPOCH,
            "max_val_batches_per_epoch": MAX_BATCHES_PER_EPOCH,
            "differential_lr": {"enabled": DIFFERENTIAL_LR_ENABLED, "backbone_lr_multiplier": BACKBONE_LR_MULTIPLIER},
        },
        "model": {
            "adapters": {"bottleneck_dim": ADAPTER_BOTTLENECK_DIM},
            "loss_balancing": {"learned_uncertainty": {"warmup_epochs": LOSS_BALANCING_WARMUP_EPOCHS}},
        },
        "paths": {
            "checkpoint_dir": paths["checkpoint_dir"].as_posix(),
            "output_dir": paths["output_dir"].as_posix(),
            "splits_dir": (RUN_DIR / "data_quality").as_posix(),
        },
        "calibration": {"output_dir": paths["calibration_dir"].as_posix()},
        "knn": {"index_dir": paths["knn_dir"].as_posix()},
    }
    overrides_args += flatten_overrides(notebook_overrides)

    write_manifest(
        paths["config_dir"] / "notebook_overrides.json",
        {"experiment_overrides": spec.get("overrides", {}), "notebook_overrides": notebook_overrides},
    )

    print(
        f"[{experiment_name} seed={seed}] training: "
        f"{'rerunning (FORCE_RERUN=True)' if checkpoint.exists() else 'running, checkpoint missing'} "
        f"(max_epochs={MAX_EPOCHS}, patience={EARLY_STOPPING_PATIENCE})"
    )
    run_command(
        [sys.executable, "-u", "scripts/train.py", "--experiment-name", experiment_name] + overrides_args,
        cwd=REPO_DIR, log_path=paths["log_dir"] / "train.log",
    )
    print(f"[{experiment_name} seed={seed}] training: checkpoint written to {checkpoint}")

    # Every checkpoint embeds the full merged config it was produced with
    # (src/training/checkpointing.py:save_checkpoint) -- extract and save it
    # alongside the notebook-level override diff above, so the *actual*
    # resolved configuration is on disk, not just the requested overrides.
    if checkpoint.exists():
        import torch as _torch

        resolved_config = _torch.load(checkpoint, map_location="cpu")["config"]
        write_manifest(paths["config_dir"] / "resolved_config.json", resolved_config)
    return paths


def calibrate_one_experiment(experiment_name, seed, paths):
    checkpoint = paths["checkpoint_dir"] / f"{experiment_name}_best_balanced_score.pt"
    validate_required_artifacts([checkpoint])

    calibration_artifact = paths["calibration_dir"] / "conformal_calibration.json"
    if calibration_artifact.exists() and not FORCE_RERUN:
        print(f"[{experiment_name} seed={seed}] calibration: skipped, artifact found ({calibration_artifact})")
        return

    print(
        f"[{experiment_name} seed={seed}] calibration: "
        f"{'rerunning (FORCE_RERUN=True)' if calibration_artifact.exists() else 'running, artifact missing'}"
    )
    run_command(
        [sys.executable, "-u", "scripts/calibrate.py", "--checkpoint", str(checkpoint)],
        cwd=REPO_DIR, log_path=paths["log_dir"] / "calibrate.log",
    )
    print(f"[{experiment_name} seed={seed}] calibration: artifact written to {calibration_artifact}")


def build_knn_one_experiment(experiment_name, seed, paths):
    checkpoint = paths["checkpoint_dir"] / f"{experiment_name}_best_balanced_score.pt"
    knn_index = paths["knn_dir"] / "knn_baseline.pkl"
    if knn_index.exists() and not FORCE_RERUN:
        print(f"[{experiment_name} seed={seed}] k-NN index: skipped, index found ({knn_index})")
        return

    print(
        f"[{experiment_name} seed={seed}] k-NN index: "
        f"{'rerunning (FORCE_RERUN=True)' if knn_index.exists() else 'running, index missing'}"
    )
    run_command(
        [sys.executable, "-u", "scripts/build_knn_index.py", "--checkpoint", str(checkpoint)],
        cwd=REPO_DIR, log_path=paths["log_dir"] / "build_knn_index.log",
    )
    print(f"[{experiment_name} seed={seed}] k-NN index: written to {knn_index}")


def evaluate_one_experiment(experiment_name, seed, paths, include_knn):
    checkpoint = paths["checkpoint_dir"] / f"{experiment_name}_best_balanced_score.pt"
    knn_index = paths["knn_dir"] / "knn_baseline.pkl"
    metrics_path = paths["output_dir"] / "metrics" / f"{experiment_name}_test_metrics.json"

    if metrics_path.exists() and not FORCE_RERUN:
        print(f"[{experiment_name} seed={seed}] evaluation: skipped, metrics found ({metrics_path})")
        return load_json(metrics_path)

    print(
        f"[{experiment_name} seed={seed}] evaluation: "
        f"{'rerunning (FORCE_RERUN=True)' if metrics_path.exists() else 'running, metrics missing'}"
    )
    eval_args = [
        sys.executable, "-u", "scripts/evaluate.py", "--checkpoint", str(checkpoint),
        "--calibration-dir", str(paths["calibration_dir"]),
        "--output-name", f"{experiment_name}_test_metrics",
    ]
    if include_knn and knn_index.exists():
        eval_args += ["--compare-knn", "--knn-path", str(knn_index)]
    run_command(eval_args, cwd=REPO_DIR, log_path=paths["log_dir"] / "evaluate.log")

    validate_required_artifacts([metrics_path])
    print(f"[{experiment_name} seed={seed}] evaluation: metrics written to {metrics_path}")
    return load_json(metrics_path)


def run_experiment_pipeline(experiment_name, seed, include_knn):
    """Run (or skip, per-stage) train -> calibrate -> [k-NN index] -> evaluate for one experiment/seed."""
    preview_paths = experiment_paths(RUN_DIR, experiment_name, seed)
    for value in preview_paths.values():
        value.mkdir(parents=True, exist_ok=True)
    print_stage_plan(experiment_name, seed, preview_paths, include_knn)

    paths = train_one_experiment(experiment_name, seed)
    calibrate_one_experiment(experiment_name, seed, paths)
    if include_knn:
        build_knn_one_experiment(experiment_name, seed, paths)
    metrics = evaluate_one_experiment(experiment_name, seed, paths, include_knn)
    return paths, metrics


def mirror_to_run_level(paths, flat_name):
    """Copy this experiment/seed's metrics/plots/checkpoints/calibration into the
    flat <RUN_DIR>/{metrics,plots,checkpoints,calibration}/ mirror, renaming
    files from their bare experiment_name prefix to flat_name. flat_name is
    the bare experiment name for the primary seed (so
    src.evaluation.final_report's discovery functions find it directly), or
    f"{experiment_name}_seed{seed}" for additional multi-seed runs (matching
    src.evaluation.comparison's seed-aggregation naming convention).
    """
    experiment_name = paths["base"].parent.name
    for sub in ("metrics", "plots"):
        src_dir = paths["output_dir"] / sub
        if not src_dir.exists():
            continue
        for file_path in src_dir.glob(f"{experiment_name}_*"):
            renamed = flat_name + file_path.name[len(experiment_name):]
            dest = RUN_DIR / sub / renamed
            dest.parent.mkdir(parents=True, exist_ok=True)
            safe_copy2(file_path, dest)
    for checkpoint_path in paths["checkpoint_dir"].glob(f"{experiment_name}_*"):
        renamed = flat_name + checkpoint_path.name[len(experiment_name):]
        dest = RUN_DIR / "checkpoints" / renamed
        dest.parent.mkdir(parents=True, exist_ok=True)
        safe_copy2(checkpoint_path, dest)
    copy_tree_merge(paths["calibration_dir"], RUN_DIR / "calibration" / flat_name)


print("Training helpers ready.")

In [ ]:
# ============================================================
# Training -- runs every selected experiment at PRIMARY_SEED. Already-
# complete ones (checkpoint + metrics already on disk) are skipped.
# ============================================================

trained_results = {}
for name in selected_experiments:
    paths, metrics = run_experiment_pipeline(name, PRIMARY_SEED, include_knn=False)
    trained_results[name] = {"paths": paths, "test_metrics": metrics}
    mirror_to_run_level(paths, flat_name=name)

    display_image(RUN_DIR / "plots" / f"{name}_training_curves.png", f"{name}: training curves")
    display_image(RUN_DIR / "plots" / f"{name}_loss_balancing.png", f"{name}: loss balancing")

    param_breakdown_path = RUN_DIR / "metrics" / f"{name}_parameter_breakdown.json"
    param_breakdown = load_json(param_breakdown_path) if param_breakdown_path.exists() else {}
    checkpoint_path = paths["checkpoint_dir"] / f"{name}_best_balanced_score.pt"

    print(f"Checkpoint: {checkpoint_path}")
    print(f"Parameter breakdown: {param_breakdown}")
    print(
        f"[{name}] age_mae={metrics.get('age_mae')} "
        f"gender_accuracy={metrics.get('gender_accuracy')} -- DONE"
    )

    sync_after_phase(f"experiment_{name}")

print(f"\nCompleted {len(trained_results)}/{len(selected_experiments)} experiment(s): {list(trained_results)}")

## 12. Experiment E -- parametric vs. k-NN baseline

Not a separate training run: builds a k-NN index on Experiment D's
embedding space (`scripts/build_knn_index.py`) and re-evaluates
Experiment D's checkpoint with `--compare-knn` (`scripts/evaluate.py`),
written under its own `exp_e_parametric_vs_knn` name so it appears as
its own row in the comparison tables and detailed report, matching
`configs/experiments.yaml`'s `exp_e_parametric_vs_knn: base_experiment:
exp_d_shared_adapters_learned_balance`.

In [ ]:
# ============================================================
# Experiment E -- reuses Experiment D's checkpoint (never retrains).
# ============================================================

if RUN_EXPERIMENT_E:
    if RESNET_EXPERIMENT not in trained_results:
        raise RuntimeError(f"{RESNET_EXPERIMENT} must complete first -- see the training section above.")

    exp_e_name = "exp_e_parametric_vs_knn"
    _d_paths = trained_results[RESNET_EXPERIMENT]["paths"]
    _d_checkpoint = _d_paths["checkpoint_dir"] / f"{RESNET_EXPERIMENT}_best_balanced_score.pt"
    _d_knn_index = _d_paths["knn_dir"] / "knn_baseline.pkl"

    build_knn_one_experiment(RESNET_EXPERIMENT, PRIMARY_SEED, _d_paths)  # skips if already built

    exp_e_metrics_path = _d_paths["output_dir"] / "metrics" / f"{exp_e_name}_test_metrics.json"
    if exp_e_metrics_path.exists() and not FORCE_RERUN:
        print(f"[exp_e] evaluation: skipped, metrics found ({exp_e_metrics_path})")
        exp_e_metrics = load_json(exp_e_metrics_path)
    else:
        run_command(
            [
                sys.executable, "-u", "scripts/evaluate.py", "--checkpoint", str(_d_checkpoint),
                "--calibration-dir", str(_d_paths["calibration_dir"]),
                "--output-name", f"{exp_e_name}_test_metrics",
                "--compare-knn", "--knn-path", str(_d_knn_index),
            ],
            cwd=REPO_DIR, log_path=_d_paths["log_dir"] / "evaluate_exp_e.log",
        )
        validate_required_artifacts([exp_e_metrics_path])
        exp_e_metrics = load_json(exp_e_metrics_path)

    trained_results[exp_e_name] = {"paths": _d_paths, "test_metrics": exp_e_metrics}
    # mirror_to_run_level globs f"{experiment_name}_*" under the ORIGINAL
    # experiment's output dir -- exp_e's files use exp_e_name as their own
    # prefix (via --output-name above), so copy them in directly instead.
    for _f in (_d_paths["output_dir"] / "metrics").glob(f"{exp_e_name}_*"):
        safe_copy2(_f, RUN_DIR / "metrics" / _f.name)
    for _f in (_d_paths["output_dir"] / "plots").glob(f"{exp_e_name}_*"):
        safe_copy2(_f, RUN_DIR / "plots" / _f.name)

    print(json.dumps({k: v for k, v in exp_e_metrics.items() if not isinstance(v, dict)}, indent=2))
    sync_after_phase("experiment_e_knn")
else:
    print("Skipped (RUN_EXPERIMENT_E=False).")

## 13. Calibration

Calibration was fit as part of the training loop above, immediately after each checkpoint was produced, using the checkpoint's own dedicated calibration split (see the overview cell's restart-safety note and `docs/data_card.md`).

## 14. Held-out test evaluation

Evaluation (against the held-out `test` split only, using each model's own calibration artifact) also ran inside the training loop above; see the printed metrics and `metrics/*_test_metrics.json`.

In [ ]:
sync_after_phase("calibration_and_evaluation")

## 15. Plots and generated reports

In [ ]:
# ============================================================
# Plain CNN vs Custom ResNet-18
# ============================================================

from src.evaluation.comparison import build_backbone_comparison_table
from src.evaluation.reports import _backbone_comparison_interpretation


def _merged_metrics(name):
    merged = dict(trained_results[name]["test_metrics"])
    for suffix in ("parameter_breakdown", "timing"):
        path = RUN_DIR / "metrics" / f"{name}_{suffix}.json"
        if path.exists():
            merged.update(load_json(path))
    return merged


cnn_vs_resnet_interpretation = None
if SMOKE_TEST:
    print(
        "SMOKE_TEST=True: skipping the formal CNN-vs-ResNet comparison "
        "table -- smoke-test metrics must not be reported as scientific "
        "findings. Re-run with SMOKE_TEST=False for a real comparison."
    )
elif CNN_EXPERIMENT in trained_results and RESNET_EXPERIMENT in trained_results:
    cnn_metrics = _merged_metrics(CNN_EXPERIMENT)
    resnet_metrics = _merged_metrics(RESNET_EXPERIMENT)

    comparison_table = build_backbone_comparison_table(cnn_metrics, resnet_metrics)
    comparison_table.to_csv(RUN_DIR / "reports" / "cnn_vs_resnet_comparison.csv", index=False)
    display(comparison_table)

    cnn_vs_resnet_interpretation = _backbone_comparison_interpretation(cnn_metrics, resnet_metrics)
    print(cnn_vs_resnet_interpretation)
    (RUN_DIR / "reports" / "cnn_vs_resnet_interpretation.txt").write_text(
        cnn_vs_resnet_interpretation, encoding="utf-8"
    )
else:
    print(
        "CNN vs ResNet comparison unavailable: both "
        f"{CNN_EXPERIMENT} and {RESNET_EXPERIMENT} must complete successfully first "
        f"(currently completed: {list(trained_results)})."
    )

## 16. Optional robustness and Grad-CAM analyses

In [ ]:
# ============================================================
# Optional: robustness (clean vs. corrupted) for the two core models
# ============================================================

robustness_targets = [
    name for name in (CNN_EXPERIMENT, PLAIN_DEEP18_EXPERIMENT, RESNET_EXPERIMENT) if name in trained_results
]

if RUN_ROBUSTNESS and robustness_targets:
    # scripts/run_robustness.py now accepts --output-dir/--calibration-dir
    # directly, writing straight into this experiment/seed's own isolated
    # directory -- no more shutil.rmtree'ing a shared outputs/robustness/
    # directory between models (which was both a collision risk and a
    # destructive workaround for the script's previous lack of an
    # --output-dir flag).
    for name in robustness_targets:
        paths = trained_results[name]["paths"]
        checkpoint = paths["checkpoint_dir"] / f"{name}_best_balanced_score.pt"
        dest = paths["base"] / "robustness"
        run_command(
            [
                sys.executable, "-u", "scripts/run_robustness.py", "--checkpoint", str(checkpoint),
                "--output-dir", str(dest), "--calibration-dir", str(paths["calibration_dir"]),
                "--max-samples", "300",
            ],
            cwd=REPO_DIR, log_path=RUN_DIR / "logs" / f"robustness_{name}.log",
        )
        print(f"[{name}] robustness results saved to {dest}")
        for metric in ("age_mae", "gender_accuracy", "abstention_rate"):
            display_image(dest / f"robustness_{metric}.png", f"{name}: robustness -- {metric} vs. corruption severity")

    # Mirror the main backbone's (ResNet) robustness results to the flat
    # <RUN_DIR>/robustness/ location that src.evaluation.final_report expects.
    if RESNET_EXPERIMENT in robustness_targets:
        copy_tree_merge(trained_results[RESNET_EXPERIMENT]["paths"]["base"] / "robustness", RUN_DIR / "robustness")
else:
    print("Robustness analysis skipped (RUN_ROBUSTNESS=False or no target experiments completed).")

In [ ]:
# ============================================================
# Optional: Grad-CAM ("model attention visualization", not proof of causality)
# ============================================================

if RUN_GRADCAM and robustness_targets:
    shared_gradcam_dir = REPO_DIR / "outputs" / "gradcam"
    for name in robustness_targets:
        checkpoint = trained_results[name]["paths"]["checkpoint_dir"] / f"{name}_best_balanced_score.pt"
        if shared_gradcam_dir.exists():
            shutil.rmtree(shared_gradcam_dir)
        run_command(
            [sys.executable, "-u", "scripts/generate_gradcam.py", "--checkpoint", str(checkpoint), "--num-samples", "12"],
            cwd=REPO_DIR, log_path=RUN_DIR / "logs" / f"gradcam_{name}.log",
        )
        dest = RUN_DIR / "gradcam" / name
        move_tree_clear(shared_gradcam_dir, dest)
        print(f"[{name}] Grad-CAM overlays (correct / incorrect / low-confidence / widest-interval / blurry) saved to {dest}")
        for path in sorted(dest.glob("*.png"))[:6]:
            display_image(path)
else:
    print("Grad-CAM analysis skipped (RUN_GRADCAM=False or no target experiments completed).")

## 17. Residual-connection ablation (backbone comparison suite)

`scripts/compare_backbones.py`'s full suite (AURC, paired bootstrap CIs,
tail-error analysis, robustness comparison if available, and a final
explicitly-conditional interpretation) across SimpleCNN vs.
PlainDeep18NoSkip vs. Custom ResNet-18 -- this is the comparison that
actually isolates whether residual connections help, holding depth/width
fixed (SimpleCNN vs. ResNet alone is only an efficiency/accuracy
trade-off, since SimpleCNN also differs in depth and width).

In [ ]:
# ============================================================
# Optional: full backbone comparison suite (SimpleCNN vs PlainDeep18NoSkip
# vs Custom ResNet-18) -- this shells out to scripts/compare_backbones.py
# exactly like every other stage above (never reimplements its analysis
# here): clean-test summary with percentiles/tail-error rates, gender and
# age selective-risk-coverage (AURC + paired bootstrap CIs), tail-error
# CDF/bucket table, an optional robustness comparison (if the per-model
# CSVs above exist), and a final, explicitly conditional "is the added
# complexity justified" interpretation that is equally capable of
# concluding against the residual architecture.
# ============================================================

if PLAIN_DEEP18_EXPERIMENT in trained_results:
    _backbone_short_names = {
        CNN_EXPERIMENT: "simple_cnn",
        PLAIN_DEEP18_EXPERIMENT: "plain_deep18_no_skip",
        RESNET_EXPERIMENT: "custom_resnet18",
    }
    _checkpoint_args, _calibration_args, _robustness_args = [], [], []
    for _exp_name, _short_name in _backbone_short_names.items():
        if _exp_name not in trained_results:
            continue
        _ckpt = trained_results[_exp_name]["paths"]["checkpoint_dir"] / f"{_exp_name}_best_balanced_score.pt"
        _checkpoint_args += ["--checkpoint", f"{_short_name}={_ckpt}"]
        _calibration_args += ["--calibration-dir", f"{_short_name}={trained_results[_exp_name]['paths']['calibration_dir']}"]
        _robustness_csv_path = RUN_DIR / "robustness" / _exp_name / "robustness_results.csv"
        if _robustness_csv_path.exists():
            _robustness_args += ["--robustness-csv", f"{_short_name}={_robustness_csv_path}"]

    backbone_comparison_dir = RUN_DIR / "reports" / "backbone_comparison"
    run_command(
        [
            sys.executable, "-u", "scripts/compare_backbones.py",
            *_checkpoint_args, *_calibration_args, *_robustness_args,
            "--resnet-name", "custom_resnet18", "--output-dir", str(backbone_comparison_dir),
        ],
        cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "compare_backbones.log",
    )
    print(f"Backbone comparison artifacts saved to {backbone_comparison_dir}")
    interpretation_path = backbone_comparison_dir / "final_interpretation.md"
    if interpretation_path.exists():
        display(Markdown(interpretation_path.read_text(encoding="utf-8")))
    sync_after_phase("backbone_comparison")
else:
    print(f"Backbone-comparison suite skipped ({PLAIN_DEEP18_EXPERIMENT} did not complete).")

## 18. Multi-seed aggregation (optional)

In [ ]:
# ============================================================
# Optional: multi-seed aggregation for the two core models
# ============================================================

from src.evaluation.comparison import aggregate_seed_metrics, build_seed_aggregate_table
from src.evaluation.reports import _df_to_md_table

multiseed_results = {}
if RUN_MULTI_SEED:
    extra_seeds = [seed for seed in SEEDS if seed != PRIMARY_SEED]
    for name in robustness_targets:
        per_seed_metrics = [trained_results[name]["test_metrics"]] if name in trained_results else []
        for seed in extra_seeds:
            paths, metrics = run_experiment_pipeline(name, seed, include_knn=False)
            per_seed_metrics.append(metrics)
            mirror_to_run_level(paths, flat_name=f"{name}_seed{seed}")
        multiseed_results[name] = per_seed_metrics

    aggregates = {name: aggregate_seed_metrics(values) for name, values in multiseed_results.items()}
    seed_table = build_seed_aggregate_table(aggregates)
    seed_table.to_csv(RUN_DIR / "reports" / "multiseed_summary.csv", index=False)
    (RUN_DIR / "reports" / "multiseed_summary.md").write_text(_df_to_md_table(seed_table), encoding="utf-8")
    display(seed_table)
else:
    print("Multi-seed aggregation skipped (RUN_MULTI_SEED=False).")

## 19. Detailed results report

In [ ]:
# ============================================================
# Detailed results report (ablation table, seed aggregation, uncertainty
# analysis, robustness, parameter/latency plots) -- reuses this run's own
# flattened metrics/plots directories, which mirror the naming convention
# src.evaluation.final_report already expects.
# ============================================================

from src.evaluation.final_report import generate_final_results_report

if SMOKE_TEST:
    print("SMOKE_TEST=True: skipping the detailed results report (integration checks only).")
else:
    detailed_report_path = RUN_DIR / "reports" / "detailed_results_report.md"
    # report_dir (not REPO_DIR) is passed here so image links are computed
    # relative to where this .md file actually lives -- RUN_DIR sits
    # entirely outside REPO_DIR on both Colab and Kaggle, so resolving
    # links against REPO_DIR would raise ValueError (path not a subpath).
    detailed_report_md = generate_final_results_report(RUN_DIR, report_dir=detailed_report_path.parent)
    detailed_report_path.write_text(detailed_report_md, encoding="utf-8")
    print(f"Detailed results report saved to {detailed_report_path}")
    display(Markdown(detailed_report_md))

## 20. Final scientific summary

In [ ]:
# ============================================================
# Final scientific summary
# ============================================================

def _build_final_summary_md():
    lines = [f"# Final Summary -- Run {RUN_ID}\n"]
    if FORCE_RERUN:
        lines.append(
            "> Full clean rerun: every metric below was generated from a "
            "newly trained model in this run directory; nothing was reused "
            "from a prior run.\n"
        )

    lines.append("## Platform and hardware\n")
    lines.append(
        f"- Platform: {platform.platform()}\n"
        f"- GPU: {gpu_info if gpu_info else 'CPU only'}\n"
        f"- Python: {sys.version.split()[0]}\n"
        f"- PyTorch: {torch.__version__}\n"
    )

    lines.append("## Git commit\n")
    lines.append(f"- SHA: `{git_sha}`\n- Branch: `{git_branch}`\n")

    lines.append("## Dataset and split\n")
    lines.append(
        f"- Split counts: {quality_report.get('split_counts')}\n"
        f"- Split file SHA-256: `{split_hash}`\n"
        f"- Subject-level splitting available: {quality_report.get('has_subject_id')}\n"
        f"- Age distribution: {quality_report.get('age_distribution')}\n"
        f"- Gender-label distribution: {quality_report.get('gender_label_distribution')}\n"
    )

    lines.append("## Selected experiments and configuration\n")
    lines.append(
        f"- Experiments: {selected_experiments}\n"
        f"- Max epochs: {MAX_EPOCHS}, early stopping patience: {EARLY_STOPPING_PATIENCE}, "
        f"primary seed: {PRIMARY_SEED}\n"
    )
    if SMOKE_TEST:
        lines.append(
            "\n**These are smoke-test results (integration check only) and must "
            "not be treated as final scientific findings.**\n"
        )

    lines.append("## Test status\n")
    lines.append(f"- Python tests: {'PASS' if tests_passed else 'FAIL'}\n")

    lines.append("## Final metrics per experiment\n")
    for name, result in trained_results.items():
        scalar_metrics = {k: v for k, v in result["test_metrics"].items() if not isinstance(v, dict)}
        lines.append(f"### {name}\n\n```\n{json.dumps(scalar_metrics, indent=2)}\n```\n")

    lines.append("## Plain CNN vs Custom ResNet-18\n")
    if cnn_vs_resnet_interpretation:
        lines.append(cnn_vs_resnet_interpretation + "\n")
    else:
        lines.append("Not available -- both exp_0 and exp_d must complete successfully.\n")

    lines.append("## Calibration methodology\n")
    lines.append(
        "This repository fits split-conformal calibration on a dedicated "
        "`calibration` split that is disjoint from both the `validation` "
        "split (used only for early stopping / checkpoint selection) and the "
        "`test` split (used only once for final evaluation) -- see "
        "docs/data_card.md. Calibration is therefore not contaminated by "
        "checkpoint-selection data. Each experiment/seed in this run was "
        "calibrated independently from its own checkpoint's predictions on "
        "the shared calibration split; no calibration artifact was reused "
        "across models. Marginal coverage does not guarantee conditional "
        "coverage for every subgroup -- see the per-age-bucket tables in "
        "`reports/detailed_results_report.md`.\n"
    )

    lines.append("## Robustness results\n")
    if RUN_ROBUSTNESS and robustness_targets:
        lines.append(f"See `robustness/` for per-experiment CSVs and plots for: {robustness_targets}.\n")
    else:
        lines.append("Not run in this session (RUN_ROBUSTNESS=False or no target experiments completed).\n")

    lines.append("## Known limitations\n")
    lines.append(
        "- Checkpoints in this repository do not save optimizer state, so "
        "this notebook's restart-safety means skipping already-complete "
        "experiments, not resuming a training run mid-epoch.\n"
        "- Conformal calibration provides marginal (population-average) "
        "coverage, not guaranteed per-subgroup coverage.\n"
        "- Gender-label predictions reflect the source dataset's own "
        "documented labels, not a determination of gender identity.\n"
        "- Robustness/Grad-CAM figures (when run) cover only the corruption "
        "types and sample counts configured here, not an exhaustive "
        "real-world robustness audit.\n"
        "- A small seed count gives a weak estimate of run-to-run variance; "
        "see reports/multiseed_summary.md's n_seeds column.\n"
    )

    lines.append("## Artifact index\n")
    lines.append(
        f"- Run directory: `{RUN_DIR}`\n"
        "- Environment manifest: `manifests/environment.json`\n"
        "- Data manifest: `manifests/data_manifest.json`\n"
        "- Test logs: `tests/pytest.log`, `tests/pytest_junit.xml`\n"
        "- Checkpoints: `checkpoints/`\n- Metrics: `metrics/`\n- Plots: `plots/`\n"
        "- Calibration: `calibration/`\n"
    )
    if not SMOKE_TEST:
        lines.append(
            "- Detailed results report: `reports/detailed_results_report.md`\n"
            "- CNN vs ResNet table: `reports/cnn_vs_resnet_comparison.csv`\n"
        )
    if PLAIN_DEEP18_EXPERIMENT in trained_results:
        lines.append("- Residual-connection ablation: `reports/backbone_comparison/`\n")
    if RUN_EXPERIMENT_E:
        lines.append("- Experiment E (parametric vs. k-NN): `metrics/exp_e_parametric_vs_knn_test_metrics.json`\n")
    if RUN_NONPARAMETRIC_BASELINES:
        lines.append("- Non-parametric baselines: `reports/nonparametric_baselines.csv`\n")
    if RUN_ROBUSTNESS:
        lines.append("- Robustness: `robustness/`\n")
    if RUN_GRADCAM:
        lines.append("- Grad-CAM: `gradcam/`\n")
    if RUN_MULTI_SEED:
        lines.append("- Multi-seed summary: `reports/multiseed_summary.csv`, `reports/multiseed_summary.md`\n")

    return "\n".join(lines)


final_summary_md = _build_final_summary_md()
(RUN_DIR / "reports" / "final_summary.md").write_text(final_summary_md, encoding="utf-8")
display(Markdown(final_summary_md))

In [ ]:
sync_after_phase("report_generation")

## 21. Non-Parametric Baselines (optional)

Fast, CPU-only baselines that never touch this project's trained multi-task
embeddings -- an "unlearned" pixel-distance baseline (Pipeline 1: flattened
raw pixels -> train-only StandardScaler -> train-only PCA -> optional
L2-norm -> k-NN / kernel method) and a frozen-ImageNet-backbone baseline
(Pipeline 2: same protocol, on a frozen, never-fine-tuned pretrained
backbone's pooled features). See `src/evaluation/nonparametric/`.

Requires the locked split (data-setup section above) but no trained
checkpoint. Hyperparameters (k / distance metric / PCA dimensionality /
L2-normalization / kernel bandwidth) are selected on the **validation**
split only; final numbers are reported on **test** only, with k-NN age
intervals conformal-calibrated on the **calibration** split (never
validation or test) -- the same 4-way protocol every other evaluation in
this notebook uses.

In [ ]:
# ============================================================
# Non-parametric baselines (raw/PCA + frozen-backbone, k-NN + KDE) --
# reuses scripts/tune_nonparametric_baselines.py (validation-only grid
# search) and scripts/evaluate_nonparametric_baselines.py (test-only final
# evaluation, calibration-split-only conformal calibration for the k-NN
# age intervals). CPU-only; does not require a GPU.
# ============================================================

if RUN_NONPARAMETRIC_BASELINES:
    run_command(
        [sys.executable, "-u", str(REPO_DIR / "scripts" / "tune_nonparametric_baselines.py")],
        cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "nonparametric_tune.log",
    )
    run_command(
        [sys.executable, "-u", str(REPO_DIR / "scripts" / "evaluate_nonparametric_baselines.py")],
        cwd=REPO_DIR, log_path=RUN_DIR / "logs" / "nonparametric_evaluate.log",
    )
    nonparametric_results_path = REPO_DIR / "outputs" / "nonparametric" / "results.csv"
    if nonparametric_results_path.exists():
        import pandas as pd
        nonparametric_results = pd.read_csv(nonparametric_results_path)
        display(nonparametric_results)
        shutil.copy2(nonparametric_results_path, RUN_DIR / "reports" / "nonparametric_baselines.csv")
    else:
        print(
            "Non-parametric baseline results were not produced -- most likely no locked split "
            "found (see the data-setup section above). See the logs above for details."
        )
    sync_after_phase("nonparametric_baselines")
else:
    print("Skipped (RUN_NONPARAMETRIC_BASELINES=False).")

## 22. Artifact persistence and final archive

In [ ]:
# ============================================================
# Final archive
# Archive contains only this run's directory: configs, manifests, logs,
# tests, checkpoints, metrics, plots, reports, calibration, and any
# robustness/Grad-CAM/k-NN outputs. Raw dataset images live under
# REPO_DIR/data/raw, entirely outside RUN_DIR, so they are never included.
# ============================================================

archive_base = RUN_DIR.parent / f"{RUN_ID}_archive"
local_archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=RUN_DIR)
print(f"Created local archive: {local_archive_path}")

if IN_COLAB and USE_GOOGLE_DRIVE:
    drive_run_dir = Path("/content/drive/MyDrive/AgeGender/runs") / RUN_ID
    copy_tree_merge(RUN_DIR, drive_run_dir)
    drive_archive_path = Path("/content/drive/MyDrive/AgeGender/runs") / Path(local_archive_path).name
    safe_copy2(local_archive_path, drive_archive_path)
    print(f"Synchronized run directory to Google Drive: {drive_run_dir}")
    print(f"Archive copied to Google Drive: {drive_archive_path}")
else:
    print("Google Drive persistence skipped (USE_GOOGLE_DRIVE=False or not running in Colab).")

print("\nRun complete.")
print(f"RUN_ID:  {RUN_ID}")
print(f"RUN_DIR: {RUN_DIR}")